<a href="https://colab.research.google.com/github/moridin04/DisasterResponseAssistant/blob/main/disasterpreparednessassistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Disaster Preparedness Assistant


- Turn hazard indicators into a **risk profile** for each location.
- Assign **risk levels** (low/medium/high) using clustering.
- Generate **recommendations** based on predicted risk.
- Detect **anomalous** locations that look unusual or extreme.
- Train **supervised models** (RandomForest and MLP) to learn from clustered risk labels.
- Export results for sharing or reporting.

## Features
- CSV loading
- Preprocessing + KMeans clustering (predicted risk)
- Recommendations per risk level
- IsolationForest anomaly detection
- RandomForest training + prediction
- MLP (Keras) training + prediction
- Export helpers (Excel & PDF)

## Dependencies (Optional)
- **PDF export** uses `fpdf`
- **MLP** uses `tensorflow`

In [ ]:
# Optional dependencies for PDF export and MLP
# !pip -q install fpdf tensorflow

## Library Imports and Configuration
### Imports
- **NumPy / Pandas** - data handling
- **scikit-learn** - preprocessing, clustering, anomaly detection, and RandomForest
- **joblib** for saving/loading trained models

### Configuration Constants
- **RISK_MAP** maps text risk labels to numeric values so the models can compute on them.
- **NON_HAZARD_COLS** lists metadata fields that should not be treated as model features.

In [ ]:
import os
from typing import Dict, Any
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, IsolationForest
import joblib

RISK_MAP = {"low": 1, "medium": 2, "high": 3}
NON_HAZARD_COLS = ["NCR", "place", "cluster", "predicted_risk", "lat", "lon", "recommendation"]

## Data Loading and Hazard Feature Identification

- **load_data**: reads the CSV file into a DataFrame.
- **identify_hazard_columns**: excludes known metadata columns, leaving hazard indicators.
- **map_risks_to_numeric**: converts textual risk labels into numeric values and coerces numeric data safely.

In [ ]:
def load_data(csv_path: str) -> pd.DataFrame:
    return pd.read_csv(csv_path)

def identify_hazard_columns(df: pd.DataFrame):
    return [c for c in df.columns if c not in NON_HAZARD_COLS]

def map_risks_to_numeric(df: pd.DataFrame, hazard_cols):
    df_num = df.copy()
    mapping_lower = {k.lower(): v for k, v in RISK_MAP.items()}
    for col in hazard_cols:
        if df_num[col].dtype == object or pd.api.types.is_categorical_dtype(df_num[col]):
            df_num[col] = (df_num[col].astype(str)
                           .str.strip()
                           .str.lower()
                           .map(mapping_lower)
                           .replace({None: np.nan}))
        else:
            df_num[col] = pd.to_numeric(df_num[col], errors='coerce')
    return df_num


## Preprocessing + KMeans Clustering

- Detecting which columns represent hazard indicators (`identify_hazard_columns`).
- Converting categorical risk labels (low/medium/high) into numeric values (`map_risks_to_numeric`).
- Handling missing values via the most frequent value in each column.
- Standardizing features so each hazard contributes evenly to the clustering.
- Applying **KMeans clustering** to group areas by similar risk patterns.

Clustering groups locations that share similar hazard characteristics. After clustering, the average hazard level
in each cluster is used to sort clusters from **lowest** to **highest** risk.

### Output Fields
- `cluster`: the cluster ID assigned by KMeans.
- `predicted_risk`: risk label inferred from the cluster ranking.
- `recommendation`: a text guideline mapped to the predicted risk.

In [ ]:
def preprocess_and_predict(df: pd.DataFrame, n_clusters: int = 3, random_state: int = 42) -> pd.DataFrame:
    if df is None or len(df) == 0:
        raise ValueError("Input DataFrame is empty or None")

    hazard_cols = identify_hazard_columns(df)
    if not hazard_cols:
        raise ValueError("No hazard columns detected in DataFrame")

    df_num = map_risks_to_numeric(df, hazard_cols)

    imputer = SimpleImputer(strategy="most_frequent")
    df_num[hazard_cols] = imputer.fit_transform(df_num[hazard_cols])

    scaler = StandardScaler()
    X = scaler.fit_transform(df_num[hazard_cols].astype(float).values)

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    clusters = kmeans.fit_predict(X)

    df = df.copy()
    df["cluster"] = clusters

    cluster_df = pd.DataFrame(df_num[hazard_cols].values)
    cluster_df["cluster"] = clusters
    cluster_means = cluster_df.groupby("cluster").mean().mean(axis=1)
    sorted_clusters = cluster_means.sort_values().index.tolist()

    labels = ["low", "medium", "high"]
    if len(sorted_clusters) != len(labels):
        labels = [f"risk_{i}" for i in range(len(sorted_clusters))]

    risk_levels = {c: lbl for c, lbl in zip(sorted_clusters, labels)}
    df["predicted_risk"] = df["cluster"].map(risk_levels)
    df["recommendation"] = df["predicted_risk"].apply(get_recommendation)
    return df


## Recommendations and Summary Helpers

- **get_recommendation**: a simple mapping from risk level to a preparedness message.
- **get_high_risk_places**: lists all NCR locations labeled as high risk.
- **get_summary_table**: groups locations by risk for a report-friendly view.

In [ ]:
def get_recommendation(risk):
    if risk is None:
        return "No risk information available."
    risk = str(risk).strip().lower()
    if risk == "low":
        return "Stay alert, monitor weather updates."
    elif risk == "medium":
        return "Prepare emergency kit and evacuation plan."
    elif risk == "high":
        return "Follow LGU evacuation orders immediately."
    else:
        return "Follow local preparedness guidance."

def get_high_risk_places(df: pd.DataFrame):
    if "predicted_risk" not in df.columns or "NCR" not in df.columns:
        return []
    return df[df["predicted_risk"].astype(str).str.lower() == "high"]["NCR"].tolist()

def get_summary_table(df: pd.DataFrame):
    if "predicted_risk" not in df.columns or "NCR" not in df.columns:
        return pd.DataFrame(columns=["predicted_risk", "NCR"])
    return df.groupby("predicted_risk")["NCR"].apply(lambda x: ", ".join(x)).reset_index()

## Filter

- **filter_by_city**: returns rows for a specific NCR location.
- **filter_by_hazard**: returns just the NCR column and one hazard column.

In [ ]:
def filter_by_city(df: pd.DataFrame, city_name: str):
    if "NCR" not in df.columns:
        return df.iloc[0:0]
    return df[df["NCR"].astype(str).str.lower() == city_name.strip().lower()]

def filter_by_hazard(df: pd.DataFrame, hazard: str):
    hazard_col = hazard.strip()
    if hazard_col not in df.columns:
        return df.iloc[0:0]
    return df[["NCR", hazard_col]].copy()

## Exporting

- **export_to_excel**: creates an `.xlsx` file for analysis and sharing.
- **export_to_pdf**: creates a printable PDF report.

In [ ]:
def export_to_excel(df: pd.DataFrame, filename: str = "risk_report.xlsx"):
    cols = list(df.columns)
    if len(cols) == 2 and cols[0] == "NCR":
        export_df = df.copy()
        export_df.columns = ["NCR Location", "Hazard Risk Level"]
    else:
        use_cols = [c for c in ["NCR", "predicted_risk", "recommendation"] if c in df.columns]
        export_df = df[use_cols].copy() if use_cols else df.copy()
        rename_map = {}
        if "NCR" in export_df.columns:
            rename_map["NCR"] = "NCR Location"
        if "predicted_risk" in export_df.columns:
            rename_map["predicted_risk"] = "Predicted Future Risk"
        if "recommendation" in export_df.columns:
            rename_map["recommendation"] = "Recommendation"
        export_df = export_df.rename(columns=rename_map)
    export_df.to_excel(filename, index=False)
    return filename

def export_to_pdf(df: pd.DataFrame, filename: str = "risk_report.pdf"):
    try:
        from fpdf import FPDF
    except Exception as e:
        raise RuntimeError("fpdf is required for PDF export: pip install fpdf") from e

    cols = list(df.columns)
    if len(cols) == 2 and cols[0] == "NCR":
        export_df = df.copy()
        export_df.columns = ["NCR Location", "Hazard Risk Level"]
    else:
        use_cols = [c for c in ["NCR", "predicted_risk", "recommendation"] if c in df.columns]
        export_df = df[use_cols].copy() if use_cols else df.copy()
        rename_map = {}
        if "NCR" in export_df.columns:
            rename_map["NCR"] = "NCR Location"
        if "predicted_risk" in export_df.columns:
            rename_map["predicted_risk"] = "Predicted Future Risk"
        if "recommendation" in export_df.columns:
            rename_map["recommendation"] = "Recommendation"
        export_df = export_df.rename(columns=rename_map)

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, txt="Disaster Risk Report in National Capital Region", ln=True, align="C")
    pdf.ln(6)

    col_width = max(40, int(180 / max(1, len(export_df.columns))))
    for header in export_df.columns:
        pdf.set_font("Arial", 'B', 10)
        pdf.cell(col_width, 8, header[:30], border=1)
    pdf.ln()

    pdf.set_font("Arial", size=10)
    for _, row in export_df.iterrows():
        for item in row:
            txt = str(item)[:60]
            pdf.cell(col_width, 8, txt, border=1)
        pdf.ln()
    pdf.output(filename)
    return filename

## Anomaly Detection

Using **IsolationForest** to identify unusual rows that deviate from the typical hazard profile.

### Detecting anomalies:
- A location with **extreme hazard values** across multiple indicators.
- **Data entry issues** (e.g., incorrect category labels).
- **Rare combinations** of hazards not common in the dataset.

IsolationForest builds random decision trees to isolate points. Anomalies are easier to isolate
and get a negative prediction. The `contamination` parameter controls how many points are
expected to be outliers.

In [ ]:
def detect_anomalies(df: pd.DataFrame, contamination: float = 0.02, random_state: int = 42) -> pd.DataFrame:
    hazard_cols = identify_hazard_columns(df)
    if not hazard_cols:
        df = df.copy()
        df["anomaly"] = False
        return df

    df_num = map_risks_to_numeric(df, hazard_cols)
    imputer = SimpleImputer(strategy="most_frequent")
    df_num[hazard_cols] = imputer.fit_transform(df_num[hazard_cols])

    scaler = StandardScaler()
    X = scaler.fit_transform(df_num[hazard_cols].astype(float).values)

    iso_model = IsolationForest(contamination=contamination, random_state=random_state)
    outliers = iso_model.fit_predict(X)

    df = df.copy()
    df["anomaly"] = outliers == -1
    return df

## RandomForest

**RandomForestClassifier** for predicting `predicted_risk` based on hazard features.
- Handles nonlinear relationships well.
- Works with mixed data and is robust to noise.
- Provides strong baseline performance for tabular data.

### Pipeline
1. Validate that `predicted_risk` exists.
2. Identify hazard features.
3. Map risk labels to numeric values and impute missing data.
4. Standardize the features.
5. Encode labels and train a supervised model.

The returned **pack** bundles the model and all preprocessing objects, so you can
apply the exact same transformations during prediction.

In [ ]:
def train_random_forest(df: pd.DataFrame, test_size: float = 0.2, random_state: int = 42,
                        n_estimators: int = 200, class_weight: str = "balanced") -> Dict[str, Any]:
    if "predicted_risk" not in df.columns:
        raise ValueError("DataFrame must include 'predicted_risk' column to train RandomForest")

    hazard_cols = identify_hazard_columns(df)
    if not hazard_cols:
        raise ValueError("No hazard columns available for training")

    df_num = map_risks_to_numeric(df, hazard_cols)
    imputer = SimpleImputer(strategy="most_frequent")
    df_num[hazard_cols] = imputer.fit_transform(df_num[hazard_cols])

    scaler = StandardScaler()
    X = scaler.fit_transform(df_num[hazard_cols].astype(float).values)

    y_raw = df["predicted_risk"].astype(str).str.strip()
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size,
        random_state=random_state, stratify=y if len(np.unique(y)) > 1 else None
    )

    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state, class_weight=class_weight)
    rf.fit(X_train, y_train)

    return {
        "model": rf,
        "hazard_cols": hazard_cols,
        "imputer": imputer,
        "scaler": scaler,
        "label_encoder": label_encoder
    }

## RandomForest: Prediction Helper

Taking a single **sample row** (dictionary of hazard values) and a trained **pack**.
It then:
- Aligns fields with the model’s hazard columns
- Converts text risk values to numeric equivalents
- Applies the stored imputer and scaler
- Predicts the risk class and maps it back to the original label

This ensures prediction logic is consistent with training.

In [ ]:
def predict_row_with_rf(sample_row: Dict[str, Any], pack: Dict[str, Any]):
    cols = pack["hazard_cols"]
    imputer = pack["imputer"]
    scaler = pack["scaler"]
    model = pack["model"]
    le = pack.get("label_encoder", None)

    sample = {c: sample_row.get(c, np.nan) for c in cols}
    sample_df = pd.DataFrame([sample])

    mapping_lower = {k.lower(): v for k, v in RISK_MAP.items()}
    for c in cols:
        if sample_df[c].dtype == object or pd.api.types.is_string_dtype(sample_df[c]):
            sample_df[c] = sample_df[c].astype(str).str.strip().str.lower().map(mapping_lower).replace({None: np.nan})

    sample_df[cols] = imputer.transform(sample_df[cols])
    Xs = scaler.transform(sample_df[cols].astype(float).values)

    pred = model.predict(Xs)
    return le.inverse_transform(pred)[0] if le is not None else int(pred[0])

## MLP - Keras

**Multi-Layer Perceptron (MLP)** neural network using TensorFlow/Keras.
- **Nonlinear modeling**: MLPs capture complex relationships between hazards that linear models may miss.
- **Flexible architecture**: You can adjust depth and width to fit the dataset.
- **Probability outputs**: Softmax output produces class probabilities for interpretability.

### Training Pipeline
1. **Dependency check**
   - Tries to import TensorFlow. If unavailable, raises a helpful error.

2. **Input validation**
   - Confirms `label_col` (default `predicted_risk`) exists.
   - Prevents silent failures from missing labels.

3. **Feature selection**
   - Uses `identify_hazard_columns` to isolate hazard indicators.
   - Ensures metadata columns like `NCR`, `cluster`, and `recommendation` are excluded.

4. **Preprocessing (critical for neural nets)**
   - **Mapping**: textual risk values → numeric values via `RISK_MAP`.
   - **Imputation**: fills missing values with the most frequent category.
   - **Standardization**: rescales features to zero mean / unit variance.
   - These steps improve stability and convergence during training.

5. **Label encoding**
   - Converts target labels (e.g., "low", "medium", "high") to integer IDs.
   - Maintains mapping for decoding predictions later.

6. **Train/test split**
   - Creates a validation set for monitoring generalization.
   - Uses stratification when multiple classes are present.

7. **Model architecture**
   - **Input layer**: matches number of hazard features.
   - **Hidden layers**: configurable via `hidden_units` (default `[64, 32]`).
   - **Dropout**: applied to reduce overfitting (default `[0.3, 0.2]`).
   - **Output layer**: softmax with `n_classes` units for multi-class classification.

8. **Training**
   - Optimizer: **Adam** (good default for tabular data).
   - Loss: **sparse categorical cross-entropy** for integer class labels.
   - Metrics: accuracy.
   - Returns training history for plotting loss/accuracy curves.

### Prediction Flow
- Aligns input columns to the model’s hazard set.
- Applies mapping, imputation, and scaling.
- Runs the model to get class probabilities.
- Picks the highest-probability class and decodes it back to the label.

### Tips
- **Small datasets**: reduce hidden units or increase dropout to avoid overfitting.
- **Imbalanced data**: consider class weights or resampling.
- **Monitoring**: inspect loss/accuracy curves from `history` to diagnose overfitting.
- **Reproducibility**: set random seeds if you want consistent training results.

In [ ]:
def train_mlp(df: pd.DataFrame, label_col: str = "predicted_risk", test_size: float = 0.2,
              random_state: int = 42, epochs: int = 30, batch_size: int = 16,
              hidden_units=[64, 32], dropout_rates=[0.3, 0.2]) -> Dict[str, Any]:
    try:
        from tensorflow import keras
        from tensorflow.keras import layers
    except Exception as e:
        raise RuntimeError("TensorFlow is required for MLP. Install with: pip install tensorflow") from e

    if label_col not in df.columns:
        raise ValueError(f"DataFrame must include '{label_col}' column to train MLP")

    hazard_cols = identify_hazard_columns(df)
    if not hazard_cols:
        raise ValueError("No hazard columns available for training")

    df_num = map_risks_to_numeric(df, hazard_cols)
    imputer = SimpleImputer(strategy="most_frequent")
    df_num[hazard_cols] = imputer.fit_transform(df_num[hazard_cols])

    scaler = StandardScaler()
    X = scaler.fit_transform(df_num[hazard_cols].astype(float).values)

    y_raw = df[label_col].astype(str).str.strip().str.lower()
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state,
        stratify=y if len(np.unique(y)) > 1 else None
    )

    n_classes = len(np.unique(y))

    model = keras.Sequential()
    model.add(layers.Input(shape=(X.shape[1],)))
    for i, units in enumerate(hidden_units):
        model.add(layers.Dense(units, activation="relu"))
        if i < len(dropout_rates):
            model.add(layers.Dropout(dropout_rates[i]))
    model.add(layers.Dense(n_classes, activation="softmax"))

    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )

    return {
        "model": model,
        "hazard_cols": hazard_cols,
        "imputer": imputer,
        "scaler": scaler,
        "label_encoder": label_encoder,
        "history": history.history
    }

def predict_row_with_mlp(sample_row: Dict[str, Any], pack: Dict[str, Any]):
    cols = pack["hazard_cols"]
    imputer = pack["imputer"]
    scaler = pack["scaler"]
    model = pack["model"]
    le = pack.get("label_encoder", None)

    sample = {c: sample_row.get(c, np.nan) for c in cols}
    sample_df = pd.DataFrame([sample])

    mapping_lower = {k.lower(): v for k, v in RISK_MAP.items()}
    for c in cols:
        if sample_df[c].dtype == object or pd.api.types.is_string_dtype(sample_df[c]):
            sample_df[c] = sample_df[c].astype(str).str.strip().str.lower().map(mapping_lower).replace({None: np.nan})

    sample_df[cols] = imputer.transform(sample_df[cols])
    Xs = scaler.transform(sample_df[cols].astype(float).values)

    probs = model.predict(Xs)
    pred_idx = int(np.argmax(probs, axis=1)[0])
    return le.inverse_transform([pred_idx])[0] if le is not None else pred_idx

## Save and Load - Model Packs
- **save_pack**: writes the pack to disk using joblib.
- **load_pack**: loads a previously saved pack.

Storing the pack ensures that predictions always use the same preprocessing steps
as training.

In [ ]:
def save_pack(pack: Dict[str, Any], filename: str):
    joblib.dump(pack, filename)
    return filename

def load_pack(filename: str) -> Dict[str, Any]:
    if not os.path.exists(filename):
        raise FileNotFoundError(filename)
    return joblib.load(filename)

#Usage

Load a CSV, run the clustering-based risk estimation, and inspect the results.

Tip: Keep hazard column names consistent across runs so model packs remain compatible.

In [ ]:
# from google.colab import files
# files.upload()

# csv_path = "/content/your_file.csv"
# df = load_data(csv_path)
# df_pred = preprocess_and_predict(df)
# df_pred.head()

Run anomaly detection on your raw dataset before or after clustering to flag outliers.

In [ ]:
# df_anom = detect_anomalies(df)
# df_anom.head()

Train Random Forest model using the predicted risk labels and save the pack for reuse.

In [ ]:
# rf_pack = train_random_forest(df_pred)
# save_pack(rf_pack, "rf_model.joblib")
# loaded_rf = load_pack("rf_model.joblib")
# predict_row_with_rf({"hazard1": "high", "hazard2": "low"}, loaded_rf)

Train MLP model, then save the pack for later predictions.

In [ ]:
# mlp_pack = train_mlp(df_pred)
# save_pack(mlp_pack, "mlp_pack.joblib")
# loaded_mlp = load_pack("mlp_pack.joblib")
# predict_row_with_mlp({"hazard1": "high", "hazard2": "low"}, loaded_mlp)

## Exporting
- **Excel** (`export_to_excel`) for easy sharing and analysis.
- **PDF** (`export_to_pdf`) for reports.

In [ ]:
# export_to_excel(df_pred, "risk_report.xlsx")
# export_to_pdf(df_pred, "risk_report.pdf")